# Test SAM-Med3D Feature Quality

This notebook evaluates whether SAM-Med3D features are discriminative for tumor classification by training a simple classification head on top of the 3D encodings.

**Motivation**: Before running the full TabPFN/LoCalPFN pipeline, we want to check if the SAM-Med3D features capture useful information for benign vs malignant classification.

**Approach**:
1. Load SAM-Med3D pretrained model
2. Add a simple classification head (Global Average Pooling + Linear)
3. Train the head (with frozen encoder) or fine-tune end-to-end
4. Evaluate on validation set

**Expected outcome**: 
- Good features → High AUC (>0.7) with frozen encoder
- Poor features → Low AUC even with fine-tuning

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from med3pipe.data.prepare import prepare_for_sam3d, split_validation, find_default_sam3d_root
from med3pipe.sam.core import load_labels_from_sheet
from med3pipe.training.classification_head import run_classification_head_experiment

## Configuration

In [ ]:
# Dataset configuration
DATASET = "gist"  # Change to 'lipo' or other dataset
CATEGORY = DATASET
CT_NAME = f"ct_{DATASET.upper()}"
DATASET_ROOT = PROJECT_ROOT / "data" / DATASET
DATASET_NAME = DATASET.upper()  # For filtering in sheet.csv
CASE_SUFFIX = "_CT"  # or "_MR" for MRI datasets

# Paths
SHEET_CSV = DATASET_ROOT / "sheet.csv"  # or PROJECT_ROOT / "sheet.csv" for unified sheet
SAM3D_CHECKPOINT = None  # Set to path of your SAM-Med3D checkpoint if available

# Training configuration
FREEZE_ENCODER = True  # Set to False to fine-tune encoder
NUM_EPOCHS = 10
BATCH_SIZE = 4
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.3
IMG_SIZE = 128

# Split configuration
SPLIT_RATIO = 0.8
SEED = 2025

# Output
OUTPUT_DIR = PROJECT_ROOT / "results" / "classification_head" / DATASET

print(f"Configuration:")
print(f"  Dataset: {DATASET}")
print(f"  Dataset root: {DATASET_ROOT}")
print(f"  Freeze encoder: {FREEZE_ENCODER}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Output: {OUTPUT_DIR}")

## Step 1: Prepare Dataset

In [ ]:
sam3d_root = find_default_sam3d_root()
print(f"SAM-Med3D root: {sam3d_root}")

prepared, paths = prepare_for_sam3d(
    dataset_root=DATASET_ROOT,
    sam3d_root=sam3d_root,
    category=CATEGORY,
    ct_name=CT_NAME,
    case_glob=None,
    max_cases=None,
)

print(f"\n✓ Prepared {prepared} cases")
print(f"  Train dir: {paths.train_root}")
print(f"  Val dir: {paths.val_root}")

## Step 2: Create Validation Split

In [ ]:
split_validation(
    paths,
    split_ratio=SPLIT_RATIO,
    seed=SEED,
    copy=True,
)

print("✓ Validation split created")

## Step 3: Load Labels

In [ ]:
df, lab_map = load_labels_from_sheet(
    sheet_csv=SHEET_CSV,
    dataset_name=DATASET_NAME,
    subject_col="Subject",
    label_col="Diagnosis_binary",
    case_suffix=CASE_SUFFIX,
)

print(f"✓ Loaded {len(lab_map)} labels")
print(f"\nClass distribution:")
print(df['label'].value_counts())
print(f"\nBenign: {(df['label'] == 0).sum()}")
print(f"Malignant: {(df['label'] == 1).sum()}")

## Step 4: Run Classification Head Experiment

In [ ]:
results = run_classification_head_experiment(
    paths=paths,
    lab_map=lab_map,
    sam3d_root=sam3d_root,
    model_type="vit_b_ori",
    checkpoint=SAM3D_CHECKPOINT,
    img_size=IMG_SIZE,
    device=None,  # Auto-detect
    freeze_encoder=FREEZE_ENCODER,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    dropout=DROPOUT,
    num_workers=2,
    output_dir=OUTPUT_DIR,
)

## Results Analysis

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Plot training curves
history = results['history']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy and AUC
axes[1].plot(history['train_acc'], label='Train Acc', marker='o')
axes[1].plot(history['val_acc'], label='Val Acc', marker='s')
axes[1].plot(history['val_auc'], label='Val AUC', marker='^')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score')
axes[1].set_title('Accuracy and AUC')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Plot saved to {OUTPUT_DIR / 'training_curves.png'}")

In [ ]:
# Final metrics
metrics = results['final_metrics']

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
print(f"Best Epoch: {results['best_epoch']}")
print(f"Best Validation AUC: {results['best_auc']:.4f}")
print(f"Final Accuracy: {metrics.accuracy:.4f}")
print(f"Final AUC: {metrics.auc:.4f}")
print(f"\nConfusion Matrix:")
print(metrics.confusion_matrix)
print(f"\nClassification Report:")
print(metrics.classification_report)
print("="*60)

In [ ]:
# Plot confusion matrix
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=metrics.confusion_matrix,
    display_labels=["Benign", "Malignant"]
)
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Confusion matrix saved to {OUTPUT_DIR / 'confusion_matrix.png'}")

In [ ]:
# ROC curve
from sklearn.metrics import roc_curve, auc

fpr, tpr, thresholds = roc_curve(metrics.targets, metrics.probabilities)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ ROC curve saved to {OUTPUT_DIR / 'roc_curve.png'}")

## Interpretation

**Good Features** (AUC > 0.7 with frozen encoder):
- SAM-Med3D learned meaningful representations
- Proceed with TabPFN/LoCalPFN pipeline
- Consider fine-tuning for better performance

**Poor Features** (AUC < 0.6 even with fine-tuning):
- SAM-Med3D features may not be suitable for this task
- Consider:
  - Different pretrained weights
  - Different feature extraction strategy
  - Alternative architectures
  - Check data quality and labels

**Moderate Features** (0.6 < AUC < 0.7):
- Features have some signal but limited
- TabPFN/LoCalPFN might help extract more from features
- Consider fine-tuning SAM-Med3D on your data